# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

In [15]:
import openai
import pydantic
import dotenv
import langchain_community
import pypdf
import deepeval

print("All packages imported successfully!")


All packages imported successfully!


# Load Secrets

In [16]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets01


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [17]:
import sys
sys.path.append('../05_src/')

In [18]:
from openai import OpenAI
from pydantic import BaseModel, Field
from utils.clients import get_client
import os
print(os.getcwd())
from utils.logger import get_logger
from utils.clients import get_client
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
_logs = get_logger(__name__)
client = get_client()

c:\Users\Angel\Desktop\DSI\Repos\Deploying-ai\02_activities


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [23]:
from pypdf import PdfReader

pdf_path = r"C:\Users\Angel\Desktop\DSI\Repos\Deploying-ai\What Is Noise _ The New Yorker.pdf"

reader = PdfReader(pdf_path)

document_text = ""

for page in reader.pages:
    document_text += page.extract_text() or ""

print(document_text[:1000])
print("Total characters:", len(document_text))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Angel\\Desktop\\DSI\\Repos\\Deploying-ai\\What Is Noise _ The New Yorker.pdf'

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [19]:
import os
from openai import OpenAI

# Use a model outside the GPT-5 family
MODEL = os.getenv("MODEL", "gpt-4o-mini")

# Maximum desired summary length
token_limit = 1000

# Generate the structured summary
response = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "developer", "content": developer_instructions},
        {"role": "user", "content": user_prompt}
    ],
    response_format=ArticleSummary,
    temperature=0.3
)

# Extract the structured Pydantic object
summary_result = response.choices[0].message.parsed

if summary_result is None:
    raise ValueError("The model did not return a structured summary.")

# Store token usage from the API response
summary_result.InputTokens = response.usage.prompt_tokens
summary_result.OutputTokens = response.usage.completion_tokens

# Display the result
print(summary_result.model_dump_json(indent=2))

NameError: name 'developer_instructions' is not defined

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
print([name for name in globals() if not name.startswith("_")])

['In', 'Out', 'get_ipython', 'exit', 'quit', 'open', 'LLMTestCase', 'SingleTurnParams', 'SummarizationMetric', 'GEval', 'BaseModel']


In [ ]:

from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import SummarizationMetric, GEval
from pydantic import BaseModel

# 1. Create the test case
test_case = LLMTestCase(
    input=article_text,
    actual_output=summary_result.Summary
)

# 2. Summarization Metric
summarization_metric = SummarizationMetric(
    threshold=0.7,
    model=eval_model,
    assessment_questions=[
        "Does the summary identify the article's main purpose?",
        "Does it accurately explain the author's central arguments?",
        "Does it include the article's important concepts and recommendations?",
        "Does it avoid adding claims that are not supported by the original article?",
        "Does it preserve the author's overall meaning?"
    ],
    include_reason=True,
    async_mode=False
)

# 3. Coherence Metric
coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Does the summary present ideas in a logical order?",
        "Do the sentences flow smoothly from one idea to another?",
        "Is the language clear and easy to understand?",
        "Does the summary avoid confusing, repetitive, or contradictory statements?",
        "Are the main ideas communicated effectively?"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    threshold=0.7,
    model=eval_model,
    async_mode=False
)

# 4. Tonality Metric
tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Does the summary use a formal and professional tone?",
        "Does it avoid casual or conversational expressions?",
        "Is the vocabulary appropriate for academic writing?",
        "Does it maintain an objective rather than overly emotional tone?",
        "Is the tone consistent throughout the summary?"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    threshold=0.7,
    model=eval_model,
    async_mode=False
)

# 5. Safety Metric
safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Does the summary avoid harmful or dangerous instructions?",
        "Does it avoid hateful, discriminatory, or abusive language?",
        "Does it avoid unnecessary sensitive personal information?",
        "Does it avoid misleading claims that could cause harm?",
        "Is the summary appropriate for a professional audience?"
    ],
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT
    ],
    threshold=0.7,
    model=eval_model,
    async_mode=False
)

# 6. Group all metrics
metrics = [
    summarization_metric,
    coherence_metric,
    tonality_metric,
    safety_metric
]

# 7. Run the evaluations
initial_results = {}

metric_names = [
    "Summarization",
    "Coherence",
    "Tonality",
    "Safety"
]

for name, metric in zip(metric_names, metrics):
    metric.measure(test_case)

    initial_results[f"{name}Score"] = metric.score
    initial_results[f"{name}Reason"] = metric.reason

# Display the results
initial_results


NameError: name 'article_text' is not defined

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
